# Language Detector
Detect user natural language and diff programming language.

Save the detected languages in `data/detected_langs/` for later analysis.

In [1]:
import os
import json
import pandas as pd
from tqdm import tqdm

FUNCTIONAL_TYPES = {
    "text",
    "details",
    "diff",
    "think",
    "read-file",
    "tool-use",
    "unknown",
    "details",
}

block_records = []

for sha in tqdm(sorted(os.listdir("../data/parsed_chats"))):
    # Skip files that are not JSON
    if not (sha.endswith(".json") or sha.endswith(".md.json")):
        continue

    file_path = f"../data/parsed_chats/{sha}"
    with open(file_path, encoding="utf-8") as f:
        chat = json.load(f)

    chat_id = (
        sha[: -len(".md.json")] if sha.endswith(".md.json") else sha[: -len(".json")]
    )

    for i, msg in enumerate(chat["messages"]):
        aggregated_blocks = []
        for block_group in msg["blocks"]:
            aggregated_blocks.extend(block_group)

        for j, block in enumerate(aggregated_blocks):
            block_type = block["type"].split(":", 1)[0]
            block_group = "func" if block_type in FUNCTIONAL_TYPES else "lang"
            block_records.append(
                {
                    "sha": chat_id,
                    "msg_role": msg["role"],
                    "msg_index": i,
                    "block_index": j,
                    "block_type": block_type,
                    "block_group": block_group,
                    "content": block["content"],
                }
            )

df_blocks = pd.DataFrame(block_records)

100%|██████████| 11655/11655 [00:15<00:00, 735.27it/s]


In [2]:
import langid
from typing import Tuple


def detect_lang(text: str) -> Tuple[str, float]:
    """Detect language using langid. Returns (lang_code, confidence)."""
    if not text or not str(text).strip():
        return "en", 1.0
    try:
        lang, score = langid.classify(text)
        return lang, float(score)
    except Exception:
        return "unknown", 0.0

## User Message Language
Use `langid` to detect natural language in user text blocks.

In [3]:
RUN_DETECT_TEXT = False

os.makedirs("../data/detected_langs", exist_ok=True)
output_path = "../data/detected_langs/all_texts_with_lang.jsonl"

if RUN_DETECT_TEXT:
    print("Detecting language for user text blocks...")
    df_text = df_blocks[
        (df_blocks["block_type"] == "text")
        & (df_blocks["msg_role"].astype(str).str.lower() == "user")
    ].copy()
    df_text = df_text.rename(columns={"content": "text"})

    if not df_text.empty:
        tqdm.pandas()
        detected = df_text["text"].progress_map(detect_lang)
        df_text["lang"] = detected.map(lambda x: x[0])
        df_text["lang_confidence"] = detected.map(lambda x: x[1])
    else:
        df_text["lang"] = pd.Series(dtype="object")
        df_text["lang_confidence"] = pd.Series(dtype="float")

    df_text = df_text[
        [
            "sha",
            "msg_role",
            "msg_index",
            "block_index",
            "lang",
            "lang_confidence",
            "text",
        ]
    ]

    df_text.to_json(
        output_path,
        index=False,
        orient="records",
        lines=True,
        force_ascii=False,
    )
else:
    print("Loading previously detected text languages...")
    df_text = pd.read_json(output_path, orient="records", lines=True)

Loading previously detected text languages...


## Diff Programming Language
Use `pygments` to detect programming language in assistant diff blocks.

In [4]:
# Detect programming language for assistant diff blocks using pygments
from pygments.lexers import guess_lexer
from pygments.util import ClassNotFound

RUN_DETECT_PL = False

os.makedirs("../data/detected_langs", exist_ok=True)
output_path_pl = "../data/detected_langs/all_diffs_with_pl.jsonl"


def detect_pl(code: str):
    if not code or not str(code).strip():
        return "unknown", 0.0
    try:
        lexer = guess_lexer(code)
        name = (lexer.name or "unknown").lower()
        return name, 1.0
    except ClassNotFound:
        return "unknown", 0.0
    except Exception:
        return "unknown", 0.0


if RUN_DETECT_PL:
    print("Detecting programming language for assistant diff blocks...")
    df_pl = df_blocks[
        (df_blocks["block_type"] == "diff")
        & (df_blocks["msg_role"].astype(str).str.lower() == "assistant")
    ].copy()

    df_pl = df_pl.rename(columns={"content": "code"})

    if not df_pl.empty:
        tqdm.pandas()
        detected = df_pl["code"].progress_map(detect_pl)
        df_pl["pl"] = detected.map(lambda x: x[0])
        df_pl["pl_confidence"] = detected.map(lambda x: x[1])
    else:
        df_pl["pl"] = pd.Series(dtype="object")
        df_pl["pl_confidence"] = pd.Series(dtype="float")

    df_pl = df_pl[
        [
            "sha",
            "msg_role",
            "msg_index",
            "block_index",
            "pl",
            "pl_confidence",
            "code",
        ]
    ]

    df_pl.to_json(
        output_path_pl,
        index=False,
        orient="records",
        lines=True,
        force_ascii=False,
    )
else:
    print("Loading previously detected programming languages...")
    df_pl = pd.read_json(output_path_pl, orient="records", lines=True)

Loading previously detected programming languages...
